# Infrastructure

In [30]:
import getpass
import os

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

## OpenRouter

In [27]:
# ChatModel

llm = ChatOpenRouter(
    model="anthropic/claude-sonnet-4.5",
    temperature=0,
    max_tokens=1024,
    max_retries=2,
)

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant"
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'I don\'t have access to real-time weather data or the internet, so I can\'t tell you the current weather in San Francisco.\n\nTo get current weather information, you could:\n- Check weather websites like weather.com or weather.gov\n- Use a search engine to search "San Francisco weather"\n- Ask a voice assistant with internet access\n- Check weather apps on your phone\n\nIs there anything else I can help you with?'}]


## Qdrant

In [1]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, SparseVectorParams, VectorParams

from uuid import uuid4
from langchain_core.documents import Document

load_dotenv()

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees Fahrenheit.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

dense_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

client = QdrantClient(path="./qdrant_storage")

existing = [c.name for c in client.get_collections().collections]
if "demo_collection" not in existing:
    client.create_collection(
        collection_name="demo_collection",
        vectors_config={"dense": VectorParams(size=1536, distance=Distance.COSINE)},
        sparse_vectors_config={"sparse": SparseVectorParams()},
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_collection",
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

documents = [document_1, document_2, document_3]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("What did I have for breakfast?")
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print()


/Users/felix/Documents/1 Project/BackendEfferentHackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


I had chocolate chip pancakes and scrambled eggs for breakfast this morning.
{'source': 'tweet', '_id': '3a2e7790-a8a3-44af-8869-429a580145c1', '_collection_name': 'demo_collection'}

The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees Fahrenheit.
{'source': 'news', '_id': 'c3cc6bb2-2fc8-4e25-9629-cd1bb6c57cf6', '_collection_name': 'demo_collection'}

Building an exciting new project with LangChain - come check it out!
{'source': 'tweet', '_id': 'b4c21f07-1137-4da9-9b07-827c6e189958', '_collection_name': 'demo_collection'}

